In [ ]:
# 정보량 적은 변수 제거 + 상관관계 제거+ 더비변수 변환
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)

# ----------------------------------
# 1. 데이터 로드 및 결측치 제거
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'

# 1-1. 결측 50% 이상 열 제거
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])

# 1-2. 최빈값으로 결측치 대체
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# ----------------------------------
# 2. 범주형 변수 더미화 (타겟 제외하고)
df[target_col] = df[target_col].astype(int)
X = df.drop(columns=[target_col])
y = df[target_col]

# 자동으로 object나 category 타입, unique 값 수가 적은 수치형 변수들을 더미화
low_cardinality_cols = X.columns[X.nunique() <= 10]  # 예: 10개 이하 unique 값
X = pd.get_dummies(X, columns=low_cardinality_cols, drop_first=True)

# ----------------------------------
# 3. 정보량 적은 변수 제거
var_thresh = VarianceThreshold(threshold=0.01)
X_var_filtered = var_thresh.fit_transform(X)
var_cols = X.columns[var_thresh.get_support()]
X_var_df = pd.DataFrame(X_var_filtered, columns=var_cols)

# ----------------------------------
# 4. 상관관계 높은 변수 제거
corr_matrix = X_var_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.9)]
X_filtered = X_var_df.drop(columns=high_corr_cols)

# ----------------------------------
# 5. Train-Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. 언더샘플링
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
real_X_train, real_y_train = rus.fit_resample(X_train_scaled, y_train)
real_X_test, real_y_test = X_test_scaled, y_test

# ----------------------------------
# 8. 모델 학습 및 평가

def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)

    return {
        "Dataset": "test",
        "Algorithm": name,
        "N": len(y_true),
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "LightGBM": LGBMClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []
for name, model in models.items():
    model.fit(real_X_train, real_y_train)
    y_pred = model.predict(real_X_test)
    y_proba = model.predict_proba(real_X_test)[:, 1]
    results.append(evaluate_model(name, real_y_test, y_pred, y_proba))

# 결과 출력
result_df = pd.DataFrame(results)
result_df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.under_sampling import RandomUnderSampler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# 1. 데이터 로드 및 전처리
df = pd.read_csv("df_concat_BA1.csv")
target_col = '기부여부'
df = df.drop(columns=df.columns[df.isnull().mean() >= 0.5])
imputer = SimpleImputer(strategy='most_frequent')
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

df[target_col] = df[target_col].astype(int)
X = df.drop(columns=[target_col])
y = df[target_col]

low_cardinality_cols = X.columns[X.nunique() <= 10]
X = pd.get_dummies(X, columns=low_cardinality_cols, drop_first=True)

# 정보량 적은 변수 제거
var_thresh = VarianceThreshold(threshold=0.01)
X_var_filtered = var_thresh.fit_transform(X)
var_cols = X.columns[var_thresh.get_support()]
X_var_df = pd.DataFrame(X_var_filtered, columns=var_cols)

# 상관관계 높은 변수 제거
corr_matrix = X_var_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.9)]
X_filtered = X_var_df.drop(columns=high_corr_cols)

# -------------------------------
# 2. 평가 함수 정의
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Algorithm": name,
        "True Positive": tp,
        "True Negative": tn,
        "False Positive": fp,
        "False Negative": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 3. 모델 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
}

# -------------------------------
# 4. KFold 훈련 및 평가 (언더샘플링 포함)
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_filtered, y)):
    X_train, X_valid = X_filtered.iloc[train_idx], X_filtered.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # Scaling
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    # Undersampling
    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_resampled, y_resampled = rus.fit_resample(X_train_scaled, y_train)

    # 각 모델 학습
    for name, model in models.items():
        model.fit(X_resampled, y_resampled)
        y_pred = model.predict(X_valid_scaled)
        y_proba = model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba))

    # CNN 학습
    cnn_model = Sequential([
        Dense(64, activation='relu', input_shape=(X_resampled.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn_model.fit(X_resampled, y_resampled, epochs=10, batch_size=32, verbose=0)

    y_proba_cnn = cnn_model.predict(X_valid_scaled).flatten()
    y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn, y_proba_cnn))

# -------------------------------
# 5. KNN 별도 평가 (언더샘플링 포함)
# 전체 데이터 스케일링 & 언더샘플링
scaler = MinMaxScaler()
X_scaled_all = scaler.fit_transform(X_filtered)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_resampled_all, y_resampled_all = rus.fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

y_pred_knn = cross_val_predict(knn, X_resampled_all, y_resampled_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_resampled_all, y_resampled_all, cv=kf, method='predict_proba')[:, 1]

results.append(evaluate_model("KNN", y_resampled_all, y_pred_knn, y_proba_knn))

# -------------------------------
# 6. 결과 출력
result_df = pd.DataFrame(results)
result_df.reset_index(drop=True, inplace=True)
result_df